# HybridBCIModel Grad-CAM Analysis
**EEGNet Block1 Grad-CAM Channel Saliency — 52 Subjects (Cho2017 / GigaDB)**

| 항목 | 값 |
|---|---|
| 모델 | HybridBCIModel (EEGNet + BiLSTM + SoftmaxFusion) |
| 레이어 | EEGNetEncoder.block1 (Temporal Conv + BN) |
| 데이터 | GigaDB 100295, 52명 LOSO |
| 방법 | Grad-CAM: ReLU( Σ_filter( mean_time( grad × act ) ) ) |
| 런타임 | T4/A100 GPU 권장 (~피험자당 1–2분) |

**실행 순서:** 위→아래로 순서대로 실행. 각 섹션 독립 실행 불가 (공유 변수 사용)

---
## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

# mne는 고품질 topomap 렌더링에 사용 (없으면 matplotlib fallback)
pip('mne')
print('패키지 설치 완료')

In [ ]:
# ────────────────────────────────────────────────────
#  ★ 여기만 수정하세요 ★
DRIVE_ROOT = '/content/drive/MyDrive/BCI_Research'
# ────────────────────────────────────────────────────

from pathlib import Path

ROOT     = Path(DRIVE_ROOT)
DATA_DIR = ROOT / 'preprocessed' / 'member_A'
CKPT_DIR = ROOT / 'results' / 'checkpoints_A'
OUT_CAM  = ROOT / 'results' / 'xai_gradcam'
CAM_DIR  = ROOT / 'results' / 'gradcam'       # per-subject NPZ 캐시

for d in [OUT_CAM, CAM_DIR, OUT_CAM / 'figures']:
    d.mkdir(parents=True, exist_ok=True)

h5_files = sorted(DATA_DIR.glob('sub-*.h5'))
pt_files = sorted(CKPT_DIR.glob('best_s*.pt'))
print(f'Data : {DATA_DIR}  존재: {DATA_DIR.exists()}')
print(f'Ckpt : {CKPT_DIR}  존재: {CKPT_DIR.exists()}')
print(f'HDF5 : {len(h5_files)}개  |  Checkpoint: {len(pt_files)}개')

if not h5_files:
    raise FileNotFoundError(f'HDF5 파일 없음: {DATA_DIR}')
if not pt_files:
    raise FileNotFoundError(f'체크포인트 없음: {CKPT_DIR}')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.interpolate import griddata
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SIDS   = list(range(1, 53))
FS     = 512
print(f'PyTorch {torch.__version__}  |  Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1. 모델 정의 (공통)

In [ ]:
CFG = {
    'n_eeg_ch': 64, 'n_emg_ch': 4, 'n_times': 2304, 'n_classes': 2,
    'emg_ds_factor': 8, 'eegnet_F1': 8, 'eegnet_D': 2, 'eegnet_kern_len': 256,
    'eegnet_dropout': 0.5, 'lstm_hidden': 128, 'lstm_layers': 2,
    'lstm_dropout': 0.3, 'clf_dropout': 0.3, 'feat_dim': 256,
}
CFG['n_times_emg'] = CFG['n_times'] // CFG['emg_ds_factor']

CH_NAMES = [
    'Fp1','AF7','AF3','F1', 'F3', 'F5', 'F7',
    'FT7','FC5','FC3','FC1','FCz','FC2','FC4',
    'FC6','FT8','T7', 'C5', 'C3', 'C1', 'Cz',
    'C2', 'C4', 'C6', 'T8', 'TP7','CP5','CP3',
    'CP1','CPz','CP2','CP4','CP6','TP8',
    'P7', 'P5', 'P3', 'P1', 'Pz', 'P2',
    'P4', 'P6', 'P8', 'PO7','PO3','POz',
    'PO4','PO8','O1', 'Oz', 'O2', 'Iz',
    'Fp2','AF8','AF4','F2', 'F4', 'F6', 'F8',
    'FT9','FT10','TP9','TP10','Fpz',
]
assert len(CH_NAMES) == 64
CH_IDX    = {ch: i for i, ch in enumerate(CH_NAMES)}
MOTOR_CHS = {'C3','C1','Cz','C2','C4','FC3','FC1','FCz','FC2','FC4',
             'CP3','CP1','CPz','CP2','CP4'}


class EEGNetEncoder(nn.Module):
    def __init__(self, n_ch, n_times, F1=8, D=2, kern_len=256, dropout=0.5, feat_dim=256):
        super().__init__()
        F2 = F1 * D
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kern_len), padding=(0, kern_len//2), bias=False),
            nn.BatchNorm2d(F1))
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F2, (n_ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(dropout))
        self.block3 = nn.Sequential(
            nn.Conv2d(F2, F2, (1, 16), padding=(0, 8), groups=F2, bias=False),
            nn.Conv2d(F2, F2, 1, bias=False),
            nn.BatchNorm2d(F2), nn.ELU(), nn.AvgPool2d((1, 8)), nn.Dropout(dropout))
        flat = self._flat(n_ch, n_times)
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(flat, feat_dim), nn.ELU())

    def _flat(self, n_ch, n_times):
        with torch.no_grad():
            x = torch.zeros(1, 1, n_ch, n_times)
            return self.block3(self.block2(self.block1(x))).numel()

    def forward(self, x):
        return self.fc(self.block3(self.block2(self.block1(x.unsqueeze(1)))))


class EMGBiLSTMEncoder(nn.Module):
    def __init__(self, n_ch=4, hidden=128, n_layers=2, dropout=0.3, feat_dim=256):
        super().__init__()
        self.lstm = nn.LSTM(n_ch, hidden, n_layers, batch_first=True, bidirectional=True,
                            dropout=dropout if n_layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden * 2)
        self.fc   = nn.Sequential(nn.Linear(hidden * 2, feat_dim), nn.ELU())

    def forward(self, x):
        out, _ = self.lstm(x.permute(0, 2, 1))
        return self.fc(self.norm(out[:, -1, :]))


class SoftmaxAttentionFusion(nn.Module):
    def __init__(self, feat_dim=256):
        super().__init__()
        self.W_eeg = nn.Linear(feat_dim, feat_dim)
        self.W_emg = nn.Linear(feat_dim, feat_dim)
        self.attn  = nn.Linear(feat_dim * 2, 2)

    def forward(self, h_eeg, h_emg):
        w = F.softmax(self.attn(torch.cat([h_eeg, h_emg], dim=-1)), dim=-1)
        return w[:, 0:1]*self.W_eeg(h_eeg) + w[:, 1:2]*self.W_emg(h_emg), w


class HybridBCIModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        fd = cfg['feat_dim']
        self.eeg_enc = EEGNetEncoder(cfg['n_eeg_ch'], cfg['n_times'],
            cfg['eegnet_F1'], cfg['eegnet_D'], cfg['eegnet_kern_len'],
            cfg['eegnet_dropout'], fd)
        self.emg_enc = EMGBiLSTMEncoder(cfg['n_emg_ch'], cfg['lstm_hidden'],
            cfg['lstm_layers'], cfg['lstm_dropout'], fd)
        self.fusion  = SoftmaxAttentionFusion(fd)
        self.clf     = nn.Sequential(nn.Linear(fd, 128), nn.ELU(),
                                     nn.Dropout(cfg['clf_dropout']),
                                     nn.Linear(128, cfg['n_classes']))

    def forward(self, eeg, emg):
        fused, w = self.fusion(self.eeg_enc(eeg), self.emg_enc(emg))
        return self.clf(fused), w


def load_model(sid, device=DEVICE):
    ckpt = CKPT_DIR / f'best_s{sid:02d}.pt'
    m = HybridBCIModel(CFG).to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    return m


def load_data(sid):
    ds = CFG['emg_ds_factor']
    with h5py.File(DATA_DIR / f'sub-{sid:02d}_member_A.h5', 'r') as f:
        eeg = f['eeg/epochs'][:].astype(np.float32)
        emg = f['emg/epochs'][:, :, ::ds].astype(np.float32)
        lbl = f['labels'][:].astype(np.int64) - 1
    return eeg, emg, lbl


# sanity check
_m = load_model(1)
_m.eval()
with torch.no_grad():
    _eeg, _emg, _ = load_data(1)
    _out, _ = _m(torch.tensor(_eeg[:2]).to(DEVICE), torch.tensor(_emg[:2]).to(DEVICE))
print(f'모델 정의 OK  |  C3={CH_IDX["C3"]}  C4={CH_IDX["C4"]}')
print(f'출력 shape: {_out.shape}  softmax: {F.softmax(_out, -1)[0].cpu().numpy().round(3)}')
del _m, _eeg, _emg, _out

---
## 2. Grad-CAM — 52명 채널 현저성 지도

`EEGNetEncoder.block1` (Temporal Conv + BN) 에 forward/backward hook을 등록.  
**공식:** `CAM[ch] = ReLU( Σ_filter( mean_time( grad × activation ) ) )`

- 각 피험자 전체 trial 사용 (배치 32)
- Left MI / Right MI 각각 역전파 후 채널별 (64,) 벡터 생성
- 정규화: [0, 1] min-max per subject
- 예상 시간: T4 기준 피험자당 ~1분 → 52명 약 50분

In [ ]:
class GradCAMHook:
    """EEGNetEncoder.block1에 hook 등록; activation + gradient 캡처."""

    def __init__(self, layer: nn.Module):
        self._act  = None
        self._grad = None
        self._fwd_h = layer.register_forward_hook(self._save_act)
        self._bwd_h = layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self._act = out.detach()           # (B, F1, 64, T')

    def _save_grad(self, module, grad_in, grad_out):
        self._grad = grad_out[0].detach()  # (B, F1, 64, T')

    def remove(self):
        self._fwd_h.remove()
        self._bwd_h.remove()

    def channel_cam(self) -> torch.Tensor:
        """(B, 64) per-channel Grad-CAM importance."""
        # (B, F1, 64, T') → mean over T' → (B, F1, 64)
        weights = (self._grad * self._act).mean(dim=-1)
        cam     = weights.sum(dim=1)   # (B, 64)
        return F.relu(cam)


print('GradCAMHook 정의 완료')

In [ ]:
BATCH_SIZE      = 32
FORCE_RECOMPUTE = False  # True로 바꾸면 캐시 무시하고 전부 재계산

cam_records  = []
failed_sids  = []

for sid in SIDS:
    h5_path    = DATA_DIR / f'sub-{sid:02d}_member_A.h5'
    ckpt_path  = CKPT_DIR / f'best_s{sid:02d}.pt'
    cache_path = CAM_DIR  / f's{sid:02d}.npz'

    if not h5_path.exists() or not ckpt_path.exists():
        print(f'[s{sid:02d}] 파일 없음 — 건너뜀')
        failed_sids.append(sid)
        continue

    # ── 캐시 로드 ──────────────────────────────────────
    if cache_path.exists() and not FORCE_RECOMPUTE:
        d = np.load(cache_path)
        cam_records.append({
            'sid':      sid,
            'left_mi':  d['left_mi'],
            'right_mi': d['right_mi'],
        })
        top_l = CH_NAMES[d['left_mi'].argmax()]
        top_r = CH_NAMES[d['right_mi'].argmax()]
        print(f'[s{sid:02d}] ↩ 캐시 로드  Left top={top_l}  Right top={top_r}')
        continue

    # ── Grad-CAM 계산 ──────────────────────────────────
    try:
        eeg, emg, lbl = load_data(sid)
        model = load_model(sid)
        model.train()  # BatchNorm을 train 모드로 (gradient 통과)

        hook = GradCAMHook(model.eeg_enc.block1)
        cam_per_class = {0: [], 1: []}
        n = len(lbl)

        for start in range(0, n, BATCH_SIZE):
            end   = min(start + BATCH_SIZE, n)
            b_eeg = torch.tensor(eeg[start:end], device=DEVICE)
            b_emg = torch.tensor(emg[start:end], device=DEVICE)

            for cls in range(2):
                model.zero_grad()
                logits, _ = model(b_eeg, b_emg)          # (B, 2)
                score = logits[:, cls].sum()             # scalar
                score.backward(retain_graph=(cls == 0))
                cam  = hook.channel_cam().cpu().numpy()  # (B, 64)
                cam_per_class[cls].append(cam)

        hook.remove()

        left_mi  = np.concatenate(cam_per_class[0], axis=0).mean(axis=0)  # (64,)
        right_mi = np.concatenate(cam_per_class[1], axis=0).mean(axis=0)

        # min-max 정규화 [0, 1]
        left_mi  = (left_mi  - left_mi.min())  / (left_mi.max()  - left_mi.min()  + 1e-8)
        right_mi = (right_mi - right_mi.min()) / (right_mi.max() - right_mi.min() + 1e-8)

        np.savez_compressed(cache_path, left_mi=left_mi, right_mi=right_mi)

        cam_records.append({'sid': sid, 'left_mi': left_mi, 'right_mi': right_mi})
        print(f'[s{sid:02d}] ✓  Left top={CH_NAMES[left_mi.argmax()]}  '
              f'Right top={CH_NAMES[right_mi.argmax()]}')

    except Exception as e:
        print(f'[s{sid:02d}] ✗  {type(e).__name__}: {e}')
        failed_sids.append(sid)
    finally:
        try:
            del model, b_eeg, b_emg
        except Exception:
            pass
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

print(f'\n완료: {len(cam_records)}/52  실패: {failed_sids if failed_sids else "없음"}')
print(f'NPZ 캐시: {CAM_DIR}')

---
## 3. 집계 & 채널 위치 생성

In [ ]:
if not cam_records:
    raise RuntimeError(
        'cam_records가 비어 있습니다.\n'
        f'실패 피험자: {failed_sids}'
    )

n_cam      = len(cam_records)
left_all   = np.stack([r['left_mi']  for r in cam_records])  # (S, 64)
right_all  = np.stack([r['right_mi'] for r in cam_records])  # (S, 64)
left_mean  = left_all.mean(0)
left_std   = left_all.std(0)
right_mean = right_all.mean(0)
right_std  = right_all.std(0)

# 피험자별 CSV
c3i, czi, c4i = CH_IDX['C3'], CH_IDX['Cz'], CH_IDX['C4']
rows = []
for r in cam_records:
    for key, cls in [('left_mi', 'Left_MI'), ('right_mi', 'Right_MI')]:
        arr  = r[key]
        top5 = np.argsort(arr)[::-1][:5]
        rows.append({
            'sid': r['sid'], 'class': cls,
            **{f'top{k+1}': CH_NAMES[i] for k, i in enumerate(top5)},
            'c3': round(float(arr[c3i]), 6),
            'c4': round(float(arr[c4i]), 6),
            'laterality': round(float(arr[c4i] - arr[c3i]), 6),  # Left MI 기준 contralateral=C4
        })
pd.DataFrame(rows).to_csv(OUT_CAM / 'gradcam_per_subject.csv', index=False)

# 그룹 평균 NPZ 저장
np.savez_compressed(
    OUT_CAM / 'gradcam_group_means.npz',
    left_mean=left_mean, left_std=left_std,
    right_mean=right_mean, right_std=right_std,
    ch_names=np.array(CH_NAMES),
    sids=np.array([r['sid'] for r in cam_records]),
)

print(f'저장 완료 ({n_cam}명)')
for cls, arr in [('Left MI', left_mean), ('Right MI', right_mean)]:
    top5 = [CH_NAMES[i] for i in np.argsort(arr)[::-1][:5]]
    print(f'[{cls}] Top-5: {top5}')
    print(f'  C3={arr[c3i]:.4f}  Cz={arr[czi]:.4f}  C4={arr[c4i]:.4f}')

In [ ]:
# 2D 두피 좌표 (azimuth-elevation 투영)
_POS_DEG = {
    'Fpz': (0,90),   'FCz': (0,36),   'Cz':  (0,0),
    'CPz': (0,-36),  'Pz': (0,-54),   'POz': (0,-72),  'Oz':  (0,-90),
    'C1':  (-18,0),  'C3': (-36,0),   'C5':  (-54,0),  'T7':  (-90,0),
    'C2':  (18,0),   'C4': (36,0),    'C6':  (54,0),   'T8':  (90,0),
    'FC1': (-18,36), 'FC3':(-36,36),  'FC5': (-54,36), 'FT7': (-90,36),
    'FC2': (18,36),  'FC4':(36,36),   'FC6': (54,36),  'FT8': (90,36),
    'CP1': (-18,-36),'CP3':(-36,-36), 'CP5': (-54,-36),'TP7': (-90,-36),
    'CP2': (18,-36), 'CP4':(36,-36),  'CP6': (54,-36), 'TP8': (90,-36),
    'F1':  (-18,54), 'F3': (-36,54),  'F5':  (-54,54), 'F7':  (-90,54),
    'AF3': (-36,72), 'AF7':(-72,72),  'Fp1': (-18,90),
    'F2':  (18,54),  'F4': (36,54),   'F6':  (54,54),  'F8':  (90,54),
    'AF4': (36,72),  'AF8':(72,72),   'Fp2': (18,90),
    'P1':  (-18,-54),'P3': (-36,-54), 'P5':  (-54,-54),'P7':  (-90,-54),
    'PO3': (-36,-72),'PO7':(-72,-72), 'O1':  (-18,-90),
    'P2':  (18,-54), 'P4': (36,-54),  'P6':  (54,-54), 'P8':  (90,-54),
    'PO4': (36,-72), 'PO8':(72,-72),  'O2':  (18,-90),
    'FT9': (-108,36),'FT10':(108,36),'TP9':(-108,-36),'TP10':(108,-36),
    'Iz':  (0,-108),
}


def _deg_to_xy(az_deg, el_deg):
    az = np.radians(az_deg)
    el = np.radians(90 - abs(el_deg))
    r  = el / (np.pi / 2)
    return float(r * np.sin(az)), float(r * np.cos(az))


def build_positions():
    pos = np.zeros((64, 2))
    for i, ch in enumerate(CH_NAMES):
        if ch in _POS_DEG:
            az, el = _POS_DEG[ch]
            pos[i] = _deg_to_xy(az, el)
    return pos


POS = build_positions()  # (64, 2)

# CSV 저장 (figure6_gradcam_topomaps.py 호환)
pos_csv = CAM_DIR / 'channel_positions.csv'
pd.DataFrame({'channel': CH_NAMES, 'x': POS[:, 0], 'y': POS[:, 1]}).to_csv(pos_csv, index=False)
print(f'채널 위치 저장: {pos_csv}')

---
## 4. 시각화

In [ ]:
# ── Figure A: 그룹 평균 Grad-CAM 토포맵 (High / All / Low performers)

# ablation CSV에서 성능 기반 그룹 분류 (없으면 fallback)
acc_csv = ROOT / 'results' / 'ablation' / 'ablation_results.csv'
try:
    acc_df    = pd.read_csv(acc_csv)
    high_sids = set(acc_df.nlargest(5, 'acc_fusion')['sid'].tolist())
    low_sids  = set(acc_df.nsmallest(5, 'acc_fusion')['sid'].tolist())
except Exception:
    high_sids = {3, 14, 41, 43, 48}   # 논문 Table 내 high-performance 피험자
    low_sids  = {1,  5,  7, 28, 40}

all_sids = {r['sid'] for r in cam_records}
high_sids &= all_sids
low_sids  &= all_sids


def group_mean(key, sids_set):
    arr = [r[key] for r in cam_records if r['sid'] in sids_set]
    return np.nanmean(arr, axis=0) if arr else np.zeros(64)


group_means = {
    'High performers': {'left_mi': group_mean('left_mi', high_sids),
                         'right_mi': group_mean('right_mi', high_sids)},
    'All subjects':    {'left_mi': left_mean, 'right_mi': right_mean},
    'Low performers':  {'left_mi': group_mean('left_mi', low_sids),
                         'right_mi': group_mean('right_mi', low_sids)},
}


def topomap_ax(ax, values, pos, cmap='RdBu_r', title=''):
    xi = np.linspace(-1.1, 1.1, 200)
    yi = np.linspace(-1.1, 1.1, 200)
    xi, yi = np.meshgrid(xi, yi)
    zi = griddata(pos, values, (xi, yi), method='cubic')
    zi[np.sqrt(xi**2 + yi**2) > 1.05] = np.nan

    im = ax.contourf(xi, yi, zi, levels=64, cmap=cmap, extend='both')
    ax.contour(xi, yi, zi, levels=10, colors='k', linewidths=0.3, alpha=0.3)
    theta = np.linspace(0, 2*np.pi, 300)
    ax.plot(np.cos(theta), np.sin(theta), 'k-', lw=1.5)
    ax.plot([0, 0.05, 0], [0.98, 1.08, 0.98], 'k-', lw=1.5)
    ax.plot([-1.03,-1.1,-1.03], [-0.08, 0, 0.08], 'k-', lw=1.2)
    ax.plot([ 1.03, 1.1, 1.03], [-0.08, 0, 0.08], 'k-', lw=1.2)
    ax.scatter(pos[:, 0], pos[:, 1], c='k', s=8, zorder=5, alpha=0.5)
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=10)
    return im


groups    = ['High performers', 'All subjects', 'Low performers']
cls_names = ['Left MI', 'Right MI']
cls_keys  = ['left_mi', 'right_mi']

all_vals = np.concatenate([v.ravel() for g in group_means.values() for v in g.values()])
vmax = float(np.percentile(np.abs(all_vals[np.isfinite(all_vals)]), 99))

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
im = None
for row, (key, cls) in enumerate(zip(cls_keys, cls_names)):
    for col, grp in enumerate(groups):
        vals = group_means[grp][key]
        n_grp = len([r for r in cam_records if r['sid'] in
                     (high_sids if grp=='High performers'
                      else (low_sids if grp=='Low performers'
                            else all_sids))])
        im = topomap_ax(axes[row, col], vals, POS, cmap='RdBu_r',
                        title=f'{grp}\n(n={n_grp})')
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=11, fontweight='bold')

if im is not None:
    cbar = fig.colorbar(im, ax=axes, orientation='vertical', shrink=0.65, pad=0.02)
    cbar.set_label('Normalized Grad-CAM activation', fontsize=9)
fig.suptitle(f'Grad-CAM EEG Channel Topomaps — HybridBCIModel EEGNet (n={n_cam})',
             fontsize=13, fontweight='bold')
fig.savefig(OUT_CAM / 'figures' / 'fig_gradcam_group.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_CAM / 'figures' / 'fig_gradcam_group.pdf', bbox_inches='tight')
plt.show()
print('토포맵 저장 완료')

In [ ]:
# ── Figure B: C3 vs C4 Grad-CAM 반구 편재화 (막대 + 개인 데이터)

left_c3  = np.array([r['left_mi'][c3i]  for r in cam_records])
left_c4  = np.array([r['left_mi'][c4i]  for r in cam_records])
right_c3 = np.array([r['right_mi'][c3i] for r in cam_records])
right_c4 = np.array([r['right_mi'][c4i] for r in cam_records])

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(4)
vals   = [left_c3,  left_c4,  right_c3,  right_c4]
colors = ['#2196F3','#2196F3','#F44336','#F44336']
alphas = [0.9, 0.5, 0.9, 0.5]
labels = ['Left MI\nC3 (ipsi)', 'Left MI\nC4 (contra)',
          'Right MI\nC3 (contra)', 'Right MI\nC4 (ipsi)']

rng = np.random.default_rng(42)
for i, (v, c, a, l) in enumerate(zip(vals, colors, alphas, labels)):
    ax.bar(x[i], v.mean(), color=c, alpha=a, yerr=v.std(), capsize=5,
           ecolor='black', label=l if i < 2 else '_nolegend_', width=0.6)
    jitter = rng.uniform(-0.15, 0.15, len(v))
    ax.scatter(x[i] + jitter, v, s=12, color=c, alpha=0.35, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Mean Grad-CAM activation (normalized)', fontsize=10)
ax.set_title(
    f'Contralateral Laterality in Grad-CAM (n={n_cam})\n'
    'Expected: Left MI → C4 > C3  |  Right MI → C3 > C4',
    fontweight='bold'
)
ax.legend(handles=[Patch(facecolor='#2196F3', label='Left MI'),
                    Patch(facecolor='#F44336', label='Right MI')])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_CAM / 'figures' / 'fig_gradcam_laterality.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_CAM / 'figures' / 'fig_gradcam_laterality.pdf', bbox_inches='tight')
plt.show()
print('반구 편재화 그래프 저장 완료')

In [ ]:
# ── Figure C: Top-20 채널 중요도 막대 그래프

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (cls, arr, std) in zip(axes, [
        ('Left MI',  left_mean,  left_std),
        ('Right MI', right_mean, right_std)]):
    idx   = np.argsort(arr)[::-1][:20]
    color = ['#4CAF50' if CH_NAMES[i] in MOTOR_CHS else '#9E9E9E' for i in idx]
    ax.barh(range(20), arr[idx][::-1],
            xerr=std[idx][::-1],
            color=color[::-1], alpha=0.85, ecolor='black', capsize=3)
    ax.set_yticks(range(20))
    ax.set_yticklabels([CH_NAMES[i] for i in idx][::-1], fontsize=9)
    ax.set_xlabel('Mean Grad-CAM activation (normalized)')
    ax.set_title(f'{cls} — Top-20 EEG Channels', fontweight='bold')
    ax.legend(handles=[Patch(color='#4CAF50', label='Motor cortex'),
                        Patch(color='#9E9E9E', label='Other')], fontsize=8)

fig.suptitle(f'EEG Channel Importance — Grad-CAM (n={n_cam})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_CAM / 'figures' / 'fig_gradcam_bar.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_CAM / 'figures' / 'fig_gradcam_bar.pdf', bbox_inches='tight')
plt.show()
print('채널 중요도 막대 그래프 저장 완료')

---
## 5. (선택) MNE 고품질 토포맵 — Figure 6 논문용

`mne`가 설치된 경우 더 정확한 두피 보간 + 공식 10-20 전극 배치 적용.  
설치 안 됐으면 이 셀만 건너뛰어도 됩니다.

In [ ]:
try:
    import mne
    print(f'MNE {mne.__version__} 사용')
    USE_MNE = True
except ImportError:
    print('MNE 없음 — matplotlib fallback (위 토포맵과 동일)')
    USE_MNE = False

if USE_MNE:
    # 표준 64채널 montage 로드
    montage    = mne.channels.make_standard_montage('biosemi64')
    info       = mne.create_info(ch_names=CH_NAMES, sfreq=512, ch_types='eeg')
    info.set_montage(montage, on_missing='warn')
    mne_pos    = np.array([info['chs'][i]['loc'][:2] for i in range(len(CH_NAMES))])

    groups    = ['High performers', 'All subjects', 'Low performers']
    cls_names = ['Left MI', 'Right MI']
    cls_keys  = ['left_mi', 'right_mi']

    all_vals = np.concatenate([v.ravel() for g in group_means.values() for v in g.values()])
    vmax     = float(np.percentile(np.abs(all_vals[np.isfinite(all_vals)]), 99))

    fig, axes = plt.subplots(2, 3, figsize=(11, 7), constrained_layout=True)
    im = None
    for row, (key, cls) in enumerate(zip(cls_keys, cls_names)):
        for col, grp in enumerate(groups):
            vals = group_means[grp][key]
            n_grp = len([r for r in cam_records if r['sid'] in
                         (high_sids if grp=='High performers'
                          else (low_sids if grp=='Low performers' else all_sids))])
            im, _ = mne.viz.plot_topomap(
                vals, info, axes=axes[row, col], show=False,
                cmap='RdBu_r', vlim=(-vmax, vmax),
                sensors=True, outlines='head', contours=6,
                extrapolate='head', image_interp='cubic',
            )
            axes[row, col].set_title(f'{grp} (n={n_grp})', fontsize=10, pad=6)
            if col == 0:
                axes[row, col].set_ylabel(cls, fontsize=11, fontweight='bold', labelpad=12)

    if im is not None:
        cbar = fig.colorbar(im, ax=axes, orientation='vertical', shrink=0.72, pad=0.025)
        cbar.set_label('Mean Grad-CAM activation (normalized)', fontsize=9)
    fig.suptitle('Class-dependent Grad-CAM Scalp Topographies\n'
                 'HybridBCIModel — EEGNet Block1',
                 fontsize=13, fontweight='bold')
    fig.savefig(OUT_CAM / 'figures' / 'figure6_gradcam_mne.png', dpi=600, bbox_inches='tight')
    fig.savefig(OUT_CAM / 'figures' / 'figure6_gradcam_mne.pdf', bbox_inches='tight')
    plt.show()
    print('MNE 논문용 Figure 6 저장 완료')

---
## 6. 결과 요약

In [ ]:
print('=' * 65)
print('  Grad-CAM Analysis 결과 요약')
print('=' * 65)

print(f'\n피험자: {n_cam}/52  실패: {failed_sids if failed_sids else "없음"}')
print()
for cls, arr, std in [
        ('Left MI',  left_mean, left_std),
        ('Right MI', right_mean, right_std)]:
    top5 = [CH_NAMES[i] for i in np.argsort(arr)[::-1][:5]]
    print(f'[{cls}]')
    print(f'  Top-5 : {top5}')
    print(f'  C3={arr[c3i]:.4f}±{std[c3i]:.4f}  '
          f'Cz={arr[czi]:.4f}±{std[czi]:.4f}  '
          f'C4={arr[c4i]:.4f}±{std[c4i]:.4f}')

print(f'\n[Laterality C3 vs C4 (mean ± std)]')
print(f'  Left MI  → C3={left_c3.mean():.4f}  C4={left_c4.mean():.4f}  '
      f'ΔC4-C3={left_c4.mean()-left_c3.mean():+.4f}')
print(f'  Right MI → C3={right_c3.mean():.4f}  C4={right_c4.mean():.4f}  '
      f'ΔC3-C4={right_c3.mean()-right_c4.mean():+.4f}')

print(f'\n[저장 위치]')
print(f'  캐시 NPZ  : {CAM_DIR}')
print(f'  집계/피규어: {OUT_CAM}')
print('=' * 65)